# ZeroLengthContactASDimplex

## Modeling

In [1]:
import xara

# Model
model = xara.Model(ndm=2, ndf=2)

# Nodes
model.node(1, (0.0, 0.0))
model.node(2, (0.0, 0.0))

# Initial fixities
model.fix(1, (1, 1))
model.fix(2, (1, 0))

2.0

In [ ]:
#
# Contact element
#
Kn = 1.0e10
Kt = 100.0
mu = 0.5

model.element(
    "ZeroLengthContactASDimplex",
    1,
    (1, 2),
    Kn,
    Kt,
    mu,
    orient=(0.0, 1.0, 0.0)
)

#
# Normal load
#
N = -10.0

model.timeSeries("Linear", 1)

model.pattern("Plain", 1, "Linear")
model.load(2, (0.0, N), pattern=1)

ZeroLengthContactASDimplex element - Implemented: Akan, OD., Petracca, M., Camata, G., Spacone, E. & Lai, CG. (2020)


## Analysis

In [3]:
#
# Static analysis: apply normal force
#
model.constraints("Transformation")
model.numberer("Plain")
model.system("FullGeneral")
model.test("NormDispIncr", 1.0e-6, 10, 0)
model.algorithm("Newton")

model.integrator("LoadControl", 1.0)
model.analysis("Static")

model.analyze(1)

# Hold the normal load constant
model.loadConst("-time", 0.0)


#
# Remove horizontal constraint
#
model.remove("sp", 2, 1)


#
# Apply horizontal imposed displacement = 1.0
#
model.pattern("Plain", 2, 1)
model.sp(2, 1, 1.0)


#
# Static analysis: impose displacement incrementally
#
model.constraints("Transformation")
model.numberer("Plain")
model.system("FullGeneral")
model.test("NormDispIncr", 1.0e-6, 10, 0)
model.algorithm("Newton")

model.integrator("LoadControl", 0.01)
model.analysis("Static")

model.analyze(100)

0

## Results

In [ ]:
#
# Check results
#
model.reactions()

reference = abs(N * mu)

RFx = abs(model.nodeResponse(2, 1, "reactionForce"))

err = abs(RFx - reference) / reference

print(f"Expected X force: {reference}")
print(f"Obtained X force: {RFx}")
print(f"Relative Error: {err * 100.0} %")

Expected X force: 5.0
Obtained X force: 4.999999999999995
Relative Error: 1.0658141036401503e-13 %
